# Nhập môn kỹ thuật Hash Map qua bài Two Sum

Bài toán xuất phát từ skeleton quen thuộc trên LeetCode:

```python
class Solution:
    def twoSum(self, nums: List[int], target: int) -> List[int]:
```

Mục tiêu của notebook này là giúp bạn hiểu **kỹ thuật dùng Hash Map / Dictionary để tra cứu nhanh phần còn thiếu**, một kỹ thuật rất hay gặp trong các bài mảng, chuỗi, đếm tần suất, tìm cặp, tìm trùng lặp.

---

## Sau notebook này, bạn sẽ nắm được

1. Bài toán Two Sum yêu cầu gì.
2. Vì sao cách duyệt 2 vòng lặp dễ hiểu nhưng chậm.
3. Ý tưởng "phần còn thiếu" là gì.
4. Cách dùng `dict` trong Python để tìm phần còn thiếu trong `O(1)` trung bình.
5. Cách xử lý số âm, số 0, số trùng nhau.
6. Mẹo nhận diện khi nào nên dùng Hash Map.
7. Bài tập vận dụng từ dễ đến nâng cao.

# 1. Bài toán Two Sum là gì?

Cho một mảng số nguyên `nums` và một số nguyên `target`.

Hãy trả về **chỉ số của 2 phần tử** sao cho tổng của chúng bằng `target`.

Ví dụ:

```python
nums = [2, 7, 11, 15]
target = 9
```

Ta thấy:

```python
2 + 7 = 9
```

Chỉ số của `2` là `0`, chỉ số của `7` là `1`.

Kết quả:

```python
[0, 1]
```

---

## Điều quan trọng

Đề bài thường yêu cầu trả về **index**, không phải giá trị.

Sai:

```python
[2, 7]
```

Đúng:

```python
[0, 1]
```

In [1]:
num = [2,7,11,15]
target = 9 
print(num[0]+num[1])
print([0,1])

9
[0, 1]


In [2]:
nums = [2, 7, 11, 15]
target = 9

print(nums[0] + nums[1])
print([0, 1])

9
[0, 1]


# 2. Cách dễ nghĩ nhất: Brute Force

Ta thử mọi cặp `(i, j)` với `i < j`.

Nếu `nums[i] + nums[j] == target` thì trả về `[i, j]`.

Ví dụ với:

```python
nums = [2, 7, 11, 15]
target = 9
```

Các cặp được thử:

| i | j | nums[i] | nums[j] | tổng |
|---|---|---------|---------|------|
| 0 | 1 | 2 | 7 | 9 |
| 0 | 2 | 2 | 11 | 13 |
| 0 | 3 | 2 | 15 | 17 |
| 1 | 2 | 7 | 11 | 18 |
| 1 | 3 | 7 | 15 | 22 |
| 2 | 3 | 11 | 15 | 26 |

Ngay cặp đầu tiên đã đúng.

In [10]:
from typing import List
def two_sum_bruteforce(nums: List[int], target:int) -> List[int]:
    n = len(nums)
    for i in range(n):
        for j in range (i+1,n):
            if nums[i] + nums[j] == target: 
                return True 
    return False
print(two_sum_bruteforce([-1,2,3,4],1))
print(two_sum_bruteforce([], 1))

True
False


In [4]:
from typing import List

def two_sum_bruteforce(nums: List[int], target: int) -> List[int]:
    n = len(nums)

    for i in range(n):
        for j in range(i + 1, n):
            if nums[i] + nums[j] == target:
                return [i, j]

    return []

print(two_sum_bruteforce([2, 7, 11, 15], 9))
print(two_sum_bruteforce([3, 2, 4], 6))
print(two_sum_bruteforce([3, 3], 6))

[0, 1]
[1, 2]
[0, 1]


## Độ phức tạp của Brute Force

Giả sử mảng có `n` phần tử.

Với mỗi `i`, ta duyệt nhiều `j` phía sau.

Số cặp cần xét xấp xỉ:

```text
n * (n - 1) / 2
```

Nên độ phức tạp thời gian là:

```text
O(n^2)
```

Nếu `n = 10_000`, số cặp có thể lên tới khoảng 50 triệu.

Vì vậy cách brute force dễ hiểu nhưng chưa tối ưu.

# 3. Ý tưởng tối ưu: tìm "phần còn thiếu"

Ta cần:

```python
nums[i] + nums[j] = target
```

Khi đang đứng ở một số `x = nums[i]`, ta cần tìm số còn lại:

```python
needed = target - x
```

Ví dụ:

```python
target = 9
x = 2
needed = 9 - 2 = 7
```

Nghĩa là:

> Nếu đã có số `7` trước đó, thì `2` và `7` tạo thành đáp án.

Đây là tư duy quan trọng nhất của Two Sum:

> Không hỏi "thử x với từng số còn lại được không?"  
> Mà hỏi "số cần ghép với x là gì, trước đó mình đã thấy nó chưa?"

In [5]:
target = 10 
x = 5
needed = target - x 
print("Target:", target)
print("x:", x)
print("Needed:", needed)

Target: 10
x: 5
Needed: 5


In [6]:
target = 9
x = 2
needed = target - x

print("target =", target)
print("x =", x)
print("needed =", needed)

target = 9
x = 2
needed = 7


# 4. Hash Map / Dictionary là gì?

Trong Python, `dict` cho phép lưu cặp:

```python
key -> value
```

Ví dụ:

```python
seen = {
    2: 0,
    7: 1
}
```

Có thể hiểu là:

```text
giá trị số -> vị trí index của nó
```

Nghĩa là:

```text
số 2 nằm ở index 0
số 7 nằm ở index 1
```

Ta dùng `dict` vì kiểm tra một key có tồn tại hay không thường rất nhanh:

```python
if needed in seen:
    ...
```

Trung bình là `O(1)`.

In [7]:
seen = {}

seen[2] = 0 
seen[7] = 1
print(seen)
print(seen[2])
print(seen[7])

print(2 in seen)
print(7 in seen)

{2: 0, 7: 1}
0
1
True
True


In [8]:
seen = {}

seen[2] = 0
seen[7] = 1

print(seen)
print(2 in seen)
print(11 in seen)
print("Index của số 7 là:", seen[7])

{2: 0, 7: 1}
True
False
Index của số 7 là: 1


# 5. Thuật toán Two Sum bằng Hash Map

Ý tưởng:

1. Tạo dictionary rỗng `seen`.
2. Duyệt từng số `x = nums[i]`.
3. Tính `needed = target - x`.
4. Nếu `needed` đã có trong `seen`, ta tìm được cặp:
   ```python
   [seen[needed], i]
   ```
5. Nếu chưa có, lưu `x` vào `seen`:
   ```python
   seen[x] = i
   ```

---

## Vì sao phải kiểm tra trước rồi mới lưu?

Ví dụ:

```python
nums = [3, 3]
target = 6
```

Khi đang ở phần tử đầu tiên `3`, nếu ta lưu trước rồi kiểm tra, có thể vô tình dùng chính nó 2 lần.

Cách an toàn:

```python
for i, x in enumerate(nums):
    needed = target - x

    if needed in seen:
        return [seen[needed], i]

    seen[x] = i
```

Nghĩa là mỗi lần xét `x`, ta chỉ ghép `x` với các số đã xuất hiện **trước nó**.

In [ ]:
from typing import List 
def two_sum_bruteforce(nums: List[int], target: int) -> List[int]: 
    

In [9]:
from typing import List
def two_sum_hashmap(nums: List[int], target: int) -> List[int]:
    seen = {}
    for i, x in enumerate(nums):
        needed = target - x
        if needed in seen: 
            return [seen[needed], i]
        seen[x] = i 
    return []
print(two_sum_hashmap([3,3,3], 6))

[0, 1]


In [10]:
def two_sum_hashmap(nums: List[int], target: int) -> List[int]:
    seen = {}

    for i, x in enumerate(nums):
        needed = target - x

        if needed in seen:
            return [seen[needed], i]

        seen[x] = i

    return []

print(two_sum_hashmap([2, 7, 11, 15], 9))
print(two_sum_hashmap([3, 2, 4], 6))
print(two_sum_hashmap([3, 3], 6))

[0, 1]
[1, 2]
[0, 1]


# 6. Mô phỏng từng bước

Xét:

```python
nums = [2, 7, 11, 15]
target = 9
```

Ban đầu:

```python
seen = {}
```

---

## Bước 1

```python
i = 0
x = 2
needed = 9 - 2 = 7
```

`7` chưa có trong `seen`.

Ta lưu:

```python
seen[2] = 0
```

Bây giờ:

```python
seen = {2: 0}
```

---

## Bước 2

```python
i = 1
x = 7
needed = 9 - 7 = 2
```

`2` đã có trong `seen`.

Vậy đáp án là:

```python
[seen[2], 1] = [0, 1]
```

In [11]:
def two_sum_trace(nums: List[int], target: int) -> List[int]:
    seen = {}

    for i, x in enumerate(nums):
        needed = target - x

        print("=" * 40)
        print(f"i = {i}")
        print(f"x = nums[{i}] = {x}")
        print(f"needed = target - x = {target} - {x} = {needed}")
        print(f"seen hiện tại = {seen}")

        if needed in seen:
            print(f"Tìm thấy needed = {needed} trong seen!")
            print(f"Index của needed là seen[{needed}] = {seen[needed]}")
            print(f"Trả về [{seen[needed]}, {i}]")
            return [seen[needed], i]

        print(f"Chưa thấy {needed}, nên lưu {x}: {i} vào seen")
        seen[x] = i

    return []

two_sum_trace([2, 7, 11, 15], 9)

i = 0
x = nums[0] = 2
needed = target - x = 9 - 2 = 7
seen hiện tại = {}
Chưa thấy 7, nên lưu 2: 0 vào seen
i = 1
x = nums[1] = 7
needed = target - x = 9 - 7 = 2
seen hiện tại = {2: 0}
Tìm thấy needed = 2 trong seen!
Index của needed là seen[2] = 0
Trả về [0, 1]


[0, 1]

# 7. Phiên bản đúng format LeetCode

Trên LeetCode, ta cần viết trong class `Solution`.

Lưu ý phải import `List` nếu chạy ở môi trường Python bình thường.

```python
from typing import List
```

Code hoàn chỉnh:

In [12]:
from typing import List

class Solution:
    def twoSum(self, nums: List[int], target: int) -> List[int]:
        seen = {}

        for i, x in enumerate(nums):
            needed = target - x

            if needed in seen:
                return [seen[needed], i]

            seen[x] = i

        return []

In [13]:
sol = Solution()

print(sol.twoSum([2, 7, 11, 15], 9))
print(sol.twoSum([3, 2, 4], 6))
print(sol.twoSum([3, 3], 6))

[0, 1]
[1, 2]
[0, 1]


# 8. Chứng minh ngắn gọn vì sao thuật toán đúng

Khi duyệt tới vị trí `i`, dictionary `seen` chứa các số đã xuất hiện ở các vị trí trước đó:

```text
0, 1, 2, ..., i - 1
```

Ta đang xét:

```python
x = nums[i]
```

Nếu tồn tại một vị trí `j < i` sao cho:

```python
nums[j] + nums[i] == target
```

thì:

```python
nums[j] == target - nums[i]
```

Mà `target - nums[i]` chính là `needed`.

Nếu `needed` nằm trong `seen`, ta lấy được index `j = seen[needed]`.

Vậy `[j, i]` là một đáp án hợp lệ.

---

## Trực giác đơn giản

Mỗi số khi đi qua đều để lại "dấu vết" trong `seen`.

Khi gặp số mới, ta hỏi:

> Trong những dấu vết trước đó, có số nào ghép với mình ra target không?

# 9. Độ phức tạp

Với cách Hash Map:

```python
for i, x in enumerate(nums):
    ...
```

Ta chỉ duyệt mảng một lần.

Mỗi lần kiểm tra:

```python
needed in seen
```

trung bình là `O(1)`.

Vì vậy:

```text
Time complexity: O(n)
Space complexity: O(n)
```

So với brute force:

| Cách làm | Thời gian | Bộ nhớ |
|---|---:|---:|
| Brute force | O(n²) | O(1) |
| Hash Map | O(n) | O(n) |

Hash Map nhanh hơn nhưng dùng thêm bộ nhớ.

# 10. Những lỗi dễ mắc

## Lỗi 1: Trả về giá trị thay vì index

Sai:

```python
return [needed, x]
```

Đúng:

```python
return [seen[needed], i]
```

---

## Lỗi 2: Dùng cùng một phần tử 2 lần

Với:

```python
nums = [3]
target = 6
```

Không được lấy `3 + 3` nếu chỉ có một số `3`.

Vì vậy ta kiểm tra `needed` trong `seen` trước, rồi mới lưu `x`.

---

## Lỗi 3: Không xử lý số trùng nhau

Ví dụ:

```python
nums = [3, 3]
target = 6
```

Cách Hash Map vẫn xử lý đúng vì:

- Gặp số `3` đầu tiên: lưu `{3: 0}`
- Gặp số `3` thứ hai: cần `3`, đã có trong `seen`
- Trả về `[0, 1]`

---

## Lỗi 4: Nghĩ số âm làm bài khác đi

Không khác.

Ví dụ:

```python
nums = [-3, 4, 3, 90]
target = 0
```

Ta có:

```python
-3 + 3 = 0
```

Đáp án là `[0, 2]`.

In [14]:
print(two_sum_hashmap([-3, 4, 3, 90], 0))
print(two_sum_hashmap([0, 4, 3, 0], 0))
print(two_sum_hashmap([-1, -2, -3, -4, -5], -8))

[0, 2]
[0, 3]
[2, 4]


# 11. Mẹo nhận diện bài nên dùng Hash Map

Bạn nên nghĩ tới `dict` / Hash Map khi thấy các dấu hiệu sau:

## Dấu hiệu 1: Cần tìm nhanh một giá trị đã xuất hiện chưa

Ví dụ:

```python
needed = target - x
if needed in seen:
    ...
```

---

## Dấu hiệu 2: Cần lưu giá trị kèm index

Ví dụ:

```python
seen[x] = i
```

---

## Dấu hiệu 3: Cần đếm tần suất

Ví dụ:

```python
freq[x] += 1
```

---

## Dấu hiệu 4: Câu hỏi có dạng "cặp", "tổng", "hiệu", "phần còn thiếu"

Ví dụ:

- Tìm 2 số có tổng bằng `target`.
- Tìm 2 số có hiệu bằng `k`.
- Kiểm tra có phần tử trùng lặp không.
- Tìm ký tự xuất hiện đầu tiên không lặp.

# 12. Kỹ thuật phụ: Counter để đếm tần suất

Không bắt buộc cho Two Sum bản index, nhưng rất hay gặp trong các bài tương tự.

Ví dụ: đếm mỗi số xuất hiện bao nhiêu lần.

In [15]:
from collections import Counter

nums = [1,1,1,2,3,3]
freq = Counter(nums)
print(freq)
print(freq[2])
print(freq[-1])
print(freq.keys())
print(freq.values())
print(freq.items())

Counter({1: 3, 3: 2, 2: 1})
1
0
dict_keys([1, 2, 3])
dict_values([3, 1, 2])
dict_items([(1, 3), (2, 1), (3, 2)])


# 13. Biến thể 1: Kiểm tra có cặp nào tổng bằng target không

Bài này không cần trả về index, chỉ cần trả về `True` hoặc `False`.

Ví dụ:

```python
nums = [2, 7, 11, 15]
target = 9
```

Có cặp `2 + 7 = 9`, nên trả về `True`.

In [24]:
def has_pair_sum(nums: List[int], target: int) -> bool:
    seen = set()
    for x in nums: 
        needed = target - x
        if needed in seen: 
            return True
        seen.add(x)
    return False 
print(has_pair_sum([-1,2,3,4],3))

True


In [16]:
def has_pair_sum(nums: List[int], target:int) -> bool: 
    seen = set()
    for x in nums: 
        needed = target - x 
        if needed in seen: 
            return True 
        seen.add(x)
    return False 

print(has_pair_sum([1,2,3,4,6], 10))

True


In [17]:
def has_pair_sum(nums: List[int], target: int) -> bool:
    seen = set()

    for x in nums:
        needed = target - x

        if needed in seen:
            return True

        seen.add(x)

    return False

print(has_pair_sum([2, 7, 11, 15], 9))
print(has_pair_sum([1, 2, 3], 10))

True
False


Ở biến thể này, ta dùng `set` thay vì `dict`.

Vì chỉ cần biết một số đã xuất hiện chưa, không cần lưu index.

```python
seen.add(x)
```

thay vì:

```python
seen[x] = i
```

# 14. Biến thể 2: Tìm tất cả cặp giá trị có tổng bằng target

Ví dụ:

```python
nums = [1, 2, 3, 4, 5]
target = 6
```

Các cặp giá trị là:

```python
(1, 5)
(2, 4)
```

Ở đây ta tìm cặp giá trị, không phải cặp index.

In [ ]:
x = 5 
target = 11
needed = target - x 
pair = (needed,x)
print(pair)

(6, 5)


In [ ]:
def all_value_pairs_sum(nums: List[int], target: int):
    seen = set()
    pairs = set()

    for x in nums:
        needed = target - x

        if needed in seen:
            pair = tuple(sorted((needed, x)))
            pairs.add(pair)

        seen.add(x)

    return sorted(pairs)

print(all_value_pairs_sum([1, 2, 3, 4, 5], 6))
print(all_value_pairs_sum([3, 3, 3, 3], 6))

[(1, 5), (2, 4)]
[(3, 3)]


# 15. Biến thể 3: Contains Duplicate

Bài toán:

Cho mảng `nums`, kiểm tra có phần tử nào xuất hiện ít nhất 2 lần không.

Ví dụ:

```python
[1, 2, 3, 1] -> True
[1, 2, 3, 4] -> False
```

Kỹ thuật giống Two Sum:

- Duyệt từng số.
- Nếu đã thấy số đó rồi thì trả về `True`.
- Nếu chưa thấy thì lưu lại.

In [6]:
from typing import List
def contains_duplicate(nums: List[int]) -> bool: 
    seen = set()
    for x in nums: 
        if x in seen: 
            return True 
        seen.add(x)
    return False 
print(contains_duplicate([1,2,3,1]))
print(contains_duplicate([1,2,3,4]))

True
False


In [ ]:
def contains_duplicate(nums: List[int]) -> bool:
    seen = set()

    for x in nums:
        if x in seen:
            return True

        seen.add(x)

    return False

print(contains_duplicate([1, 2, 3, 1]))
print(contains_duplicate([1, 2, 3, 4]))

True
False


# 16. Template tư duy Hash Map

Khi gặp một bài có vẻ dùng Hash Map, hãy thử đi theo khung này:

```python
seen = {}

for i, x in enumerate(nums):
    # 1. Tính thứ mình đang cần tìm
    needed = ...

    # 2. Kiểm tra nó đã xuất hiện chưa
    if needed in seen:
        return ...

    # 3. Nếu chưa, lưu thông tin hiện tại
    seen[x] = i
```

Với bài đếm tần suất:

```python
freq = {}

for x in nums:
    if x not in freq:
        freq[x] = 0
    freq[x] += 1
```

Hoặc dùng `Counter`:

```python
from collections import Counter
freq = Counter(nums)
```

# 17. Bài tập vận dụng

## Bài 1 — Two Sum cơ bản

Hoàn thiện hàm sau:

```python
def two_sum(nums, target):
    ...
```

Ví dụ:

```python
two_sum([2, 7, 11, 15], 9) -> [0, 1]
two_sum([3, 2, 4], 6) -> [1, 2]
```

In [ ]:
def exercise_1_two_sum(nums: List[int], target: int) -> List[int]:
    # TODO:
    # Tạo dictionary seen
    # Duyệt từng phần tử
    # Tính needed = target - x
    # Nếu needed có trong seen thì trả về index
    # Nếu chưa có thì lưu x vào seen
    pass


# Hãy bỏ comment để test sau khi làm xong
# print(exercise_1_two_sum([2, 7, 11, 15], 9))
# print(exercise_1_two_sum([3, 2, 4], 6))

## Bài 2 — Có cặp tổng bằng target không?

Viết hàm trả về `True` nếu tồn tại 2 số có tổng bằng `target`, ngược lại trả về `False`.

Ví dụ:

```python
has_pair_sum([1, 2, 3, 4], 7) -> True
has_pair_sum([1, 2, 3, 4], 10) -> False
```

In [ ]:
def exercise_2_has_pair_sum(nums: List[int], target: int) -> bool:
    # TODO:
    # Gợi ý: dùng set là đủ vì không cần index
    pass


# print(exercise_2_has_pair_sum([1, 2, 3, 4], 7))
# print(exercise_2_has_pair_sum([1, 2, 3, 4], 10))

## Bài 3 — Đếm tần suất

Viết hàm nhận vào mảng `nums`, trả về dictionary đếm số lần xuất hiện của mỗi phần tử.

Ví dụ:

```python
count_frequency([1, 2, 2, 3, 3, 3])
```

Kết quả:

```python
{1: 1, 2: 2, 3: 3}
```

In [ ]:
def exercise_3_count_frequency(nums: List[int]) -> dict:
    # TODO:
    # Gợi ý:
    # freq = {}
    # for x in nums:
    #     ...
    pass


# print(exercise_3_count_frequency([1, 2, 2, 3, 3, 3]))

## Bài 4 — Tìm số xuất hiện đầu tiên bị lặp lại

Viết hàm trả về số đầu tiên mà khi duyệt từ trái sang phải, ta gặp nó lần thứ hai.

Ví dụ:

```python
first_repeated([2, 5, 1, 2, 3, 5]) -> 2
first_repeated([1, 2, 3]) -> None
```

In [ ]:
def exercise_4_first_repeated(nums: List[int]):
    # TODO:
    # Gợi ý:
    # Dùng set seen
    # Nếu x đã có trong seen thì return x
    # Nếu chưa có thì thêm x vào seen
    pass


# print(exercise_4_first_repeated([2, 5, 1, 2, 3, 5]))
# print(exercise_4_first_repeated([1, 2, 3]))

## Bài 5 — Two Sum nhưng trả về tất cả cặp index

Viết hàm trả về tất cả cặp index `[i, j]` sao cho:

```python
nums[i] + nums[j] == target
i < j
```

Ví dụ:

```python
all_two_sum_indices([1, 5, 1, 5], 6)
```

Có thể trả về:

```python
[[0, 1], [0, 3], [1, 2], [2, 3]]
```

Gợi ý:

- Dictionary không chỉ lưu một index.
- Có thể lưu:
  ```python
  seen[value] = [list các index đã gặp]
  ```

In [ ]:
def exercise_5_all_two_sum_indices(nums: List[int], target: int) -> List[List[int]]:
    # TODO:
    # Gợi ý:
    # seen = {}
    # for i, x in enumerate(nums):
    #     needed = target - x
    #     nếu needed trong seen:
    #         với mỗi j trong seen[needed], thêm [j, i] vào answer
    #     lưu i vào seen[x]
    pass


# print(exercise_5_all_two_sum_indices([1, 5, 1, 5], 6))

# 18. Đáp án tham khảo

Bạn nên tự làm trước khi mở phần này.

---

## Đáp án Bài 1

In [ ]:
def answer_1_two_sum(nums: List[int], target: int) -> List[int]:
    seen = {}

    for i, x in enumerate(nums):
        needed = target - x

        if needed in seen:
            return [seen[needed], i]

        seen[x] = i

    return []

print(answer_1_two_sum([2, 7, 11, 15], 9))
print(answer_1_two_sum([3, 2, 4], 6))

[0, 1]
[1, 2]


## Đáp án Bài 2

In [ ]:
def answer_2_has_pair_sum(nums: List[int], target: int) -> bool:
    seen = set()

    for x in nums:
        needed = target - x

        if needed in seen:
            return True

        seen.add(x)

    return False

print(answer_2_has_pair_sum([1, 2, 3, 4], 7))
print(answer_2_has_pair_sum([1, 2, 3, 4], 10))

True
False


## Đáp án Bài 3

In [ ]:
def answer_3_count_frequency(nums: List[int]) -> dict:
    freq = {}

    for x in nums:
        if x not in freq:
            freq[x] = 0
        freq[x] += 1

    return freq

print(answer_3_count_frequency([1, 2, 2, 3, 3, 3]))

{1: 1, 2: 2, 3: 3}


## Đáp án Bài 4

In [ ]:
def answer_4_first_repeated(nums: List[int]):
    seen = set()

    for x in nums:
        if x in seen:
            return x

        seen.add(x)

    return None

print(answer_4_first_repeated([2, 5, 1, 2, 3, 5]))
print(answer_4_first_repeated([1, 2, 3]))

2
None


## Đáp án Bài 5

In [ ]:
def answer_5_all_two_sum_indices(nums: List[int], target: int) -> List[List[int]]:
    seen = {}
    answer = []

    for i, x in enumerate(nums):
        needed = target - x

        if needed in seen:
            for j in seen[needed]:
                answer.append([j, i])

        if x not in seen:
            seen[x] = []
        seen[x].append(i)

    return answer

print(answer_5_all_two_sum_indices([1, 5, 1, 5], 6))

[[0, 1], [1, 2], [0, 3], [2, 3]]


# 19. Tổng kết

Bài Two Sum là bài nhập môn rất tốt cho kỹ thuật Hash Map.

Cốt lõi của kỹ thuật này là:

```python
needed = target - current_value
```

Sau đó hỏi:

```python
needed đã xuất hiện trước đó chưa?
```

Nếu có, ta tìm được đáp án ngay.

---

## Công thức nhớ nhanh

```python
seen = {}

for i, x in enumerate(nums):
    needed = target - x

    if needed in seen:
        return [seen[needed], i]

    seen[x] = i
```

---

## Khi nào nên nghĩ tới Hash Map?

Khi bài toán có các cụm ý:

- "đã xuất hiện chưa"
- "tìm nhanh"
- "đếm số lần"
- "tìm cặp"
- "tổng bằng target"
- "phần còn thiếu"
- "giá trị đi kèm index"

Nếu thấy brute force phải dùng 2 vòng lặp để tìm một giá trị, hãy thử nghĩ:

> Mình có thể lưu những thứ đã gặp vào `dict` hoặc `set` để tra nhanh không?